# Career matcher training

Fine-tunes a small sentence-transformer so that a student's profile ends up close to the description of the career that suits them. Once trained it can rank all 452 careers for a profile, including careers it never saw during training, because it compares the profile with each career's description rather than picking from a fixed list of labels.

**Before running:** Runtime > Change runtime type > T4 GPU. A full run takes roughly 20 to 40 minutes on the free GPU.

You will be asked for `career_coach_evaluation_data.zip` (the same data file used by the evaluation notebook). If you also upload `career_retriever_model.zip`, training starts from the project's own retriever; otherwise it starts from `all-MiniLM-L6-v2`.

In [ ]:
import os, sys, zipfile, re, time, json, random
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SEED = 42
DAMAGE = 0.5            # share of each test profile that is damaged, same as the evaluation notebook
EPOCHS = 3
BATCH_SIZE = 64
LEARNING_RATE = 2e-5
WARMUP_SHARE = 0.1
MAX_TOKENS = 256

if IN_COLAB:
    !pip -q install "sentence-transformers>=3.0"
    from google.colab import files
    print("Upload career_coach_evaluation_data.zip (and career_retriever_model.zip if you have it).")
    for name in files.upload():
        zipfile.ZipFile(name).extractall("/content/project")
    PROJECT = Path("/content/project")
else:
    PROJECT = Path(os.environ.get("AICAREERCOACH_DIR", Path.cwd().parents[1]))

CAREERS = PROJECT / "ml" / "data" / "careers"
OWN_RETRIEVER = PROJECT / "ml" / "models" / "career-retriever"
BASE_MODEL = str(OWN_RETRIEVER) if (OWN_RETRIEVER / "modules.json").is_file() else "sentence-transformers/all-MiniLM-L6-v2"
OUT_DIR = PROJECT / "ml" / "models" / "career-matcher"
RESULTS = PROJECT / "ml" / "metrics" / "career_matcher"
RESULTS.mkdir(parents=True, exist_ok=True)
print("Starting from:", BASE_MODEL)

import numpy as np, pandas as pd, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## Data

Each career gets one text built from the catalogue: title, category, the skills it needs, its description and its main tasks. Each student profile becomes a short plain-text summary. Test profiles are damaged the same way as in the evaluation notebook (items dropped, unrelated skills added, fields left empty) so the numbers are comparable.

In [ ]:
read = lambda name: pd.read_csv(CAREERS / name, encoding="utf-8-sig").fillna("")
catalog = read("career_catalog.csv")
skills_table = read("career_skills.csv")
tasks_table = read("career_tasks.csv")
profiles = read("career_training_data.csv")
train = profiles[profiles["split"] == "train"].reset_index(drop=True)
validation = profiles[profiles["split"] == "validation"].reset_index(drop=True)
test = profiles[profiles["split"] == "test"].reset_index(drop=True)
print(f"{len(train)} training, {len(validation)} validation and {len(test)} test profiles; {len(catalog)} careers")

skills_by_career = skills_table.sort_values("importance", ascending=False).groupby("career_id")["skill"].apply(list).to_dict()
tasks_by_career = tasks_table.groupby("career_id")["task"].apply(list).to_dict()
category_of = catalog.set_index("career_id")["category"].to_dict()
careers_in_category = catalog.groupby("category")["career_id"].apply(list).to_dict()
all_careers = catalog["career_id"].tolist()

def career_text(row):
    return (f"{row.role_title}. {row.category}. Skills: {', '.join(skills_by_career.get(row.career_id, []))}. "
            f"{row.description} Tasks: {' '.join(tasks_by_career.get(row.career_id, []))}")

career_ids = sorted(train["target_career_id"].unique())
career_index = {c: i for i, c in enumerate(career_ids)}
docs = catalog.set_index("career_id").loc[career_ids].reset_index()
career_texts = [career_text(r) for r in docs.itertuples()]

def profile_to_text(row):
    parts = [("Course", row.course), ("Specialisation", row.specialisation), ("Skills", row.skills),
             ("Interests", row.interests), ("Certifications", row.certifications), ("Preferred industry", row.preferred_industry)]
    return ". ".join(f"{label}: {value.replace(';', ',')}" for label, value in parts if str(value).strip())

def add_noise(frame, level, seed=SEED):
    """Same damage as the evaluation notebook."""
    rng = np.random.default_rng(seed); noisy = frame.copy()
    for column in ("skills", "interests"):
        values = []
        for text in noisy[column]:
            items = [i.strip() for i in text.split(";") if i.strip()]
            values.append("; ".join([i for i in items if rng.random() >= level] or items[:1]))
        noisy[column] = values
    out = []
    for skills, cid in zip(noisy["skills"], noisy["target_career_id"]):
        items = [i for i in skills.split("; ") if i]
        neighbours = [o for o in careers_in_category[category_of[cid]] if o != cid]
        for _ in range(int(round(level * max(len(items), 3)))):
            pool = neighbours if rng.random() < 0.7 else all_careers
            borrowed = skills_table.loc[skills_table["career_id"] == pool[rng.integers(len(pool))], "skill"].tolist()
            if borrowed: items.append(borrowed[rng.integers(len(borrowed))])
        out.append("; ".join(items))
    noisy["skills"] = out
    for column in ("course", "specialisation", "certifications", "preferred_industry"):
        noisy.loc[rng.random(len(noisy)) < level, column] = ""
    return noisy

# One career in five is kept out of training to check how the model copes with careers it has never seen.
new_careers = set(np.random.default_rng(SEED).choice(career_ids, size=len(career_ids) // 5, replace=False))
print(f"{len(new_careers)} careers held back as 'new'")

## Training

Each training example is a (profile, career text) pair. The loss is MultipleNegativesRankingLoss: in every batch the model learns to put each profile closer to its own career than to the other careers in the batch. Every profile is used twice per epoch, once as written and once damaged, so the model also learns to cope with incomplete profiles.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
try:
    from sentence_transformers.sentence_transformer.datasets import NoDuplicatesDataLoader
except ImportError:
    from sentence_transformers.datasets import NoDuplicatesDataLoader

def training_pairs(frame, skip=()):
    rows = frame[~frame["target_career_id"].isin(skip)]
    damaged = add_noise(rows, DAMAGE, seed=SEED + 7)
    pairs = []
    for version in (rows, damaged):
        for r in version.itertuples():
            pairs.append(InputExample(texts=[profile_to_text(r), career_texts[career_index[r.target_career_id]]]))
    random.Random(SEED).shuffle(pairs)
    return pairs

def train_matcher(skip=(), epochs=EPOCHS):
    model = SentenceTransformer(BASE_MODEL, device=DEVICE)
    model.max_seq_length = MAX_TOKENS
    pairs = training_pairs(train, skip)
    loader = NoDuplicatesDataLoader(pairs, batch_size=BATCH_SIZE)
    loss = losses.MultipleNegativesRankingLoss(model)
    steps = len(loader) * epochs
    start = time.time()
    model.fit(train_objectives=[(loader, loss)], epochs=epochs, warmup_steps=int(steps * WARMUP_SHARE),
              optimizer_params={"lr": LEARNING_RATE}, show_progress_bar=True)
    print(f"Trained on {len(pairs)} pairs in {(time.time() - start) / 60:.1f} minutes")
    return model

def rank_scores(model, frame):
    doc_vectors = model.encode(career_texts, batch_size=128, normalize_embeddings=True, convert_to_numpy=True)
    profile_vectors = model.encode([profile_to_text(r) for r in frame.itertuples()], batch_size=128,
                                   normalize_embeddings=True, convert_to_numpy=True)
    return profile_vectors @ doc_vectors.T

def hit_and_mrr(scores, frame):
    gold = np.array([career_index[c] for c in frame["target_career_id"]])
    ranks = (scores > scores[np.arange(len(gold)), gold][:, None]).sum(axis=1) + 1
    return round(float((ranks == 1).mean() * 100), 1), round(float((1 / ranks).mean() * 100), 1)

base = SentenceTransformer(BASE_MODEL, device=DEVICE); base.max_seq_length = MAX_TOKENS
held_out_model = train_matcher(skip=new_careers)

## Results

The matcher is compared with the starting model before fine-tuning and with the project's Logistic Regression classifier (TF-IDF features, the same settings as the backend). Hit@1 is the share of students whose career comes first; MRR also gives credit when it comes second, third and so on.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def tagged(frame):
    cols = ["course", "specialisation", "skills", "interests", "experience_level", "certifications", "preferred_industry"]
    return frame[cols].astype(str).apply(lambda r: " ".join(
        f"{c.split('_')[0]}_{re.sub(r'[^a-z0-9]+', '_', i.strip().lower())} {i}" for c in cols for i in r[c].split(";") if i.strip()), axis=1)

def lr_scores(model, vec, frame):
    out = np.zeros((len(frame), len(career_ids)))
    out[:, [career_index[c] for c in model.classes_]] = model.predict_proba(vec.transform(tagged(frame)))
    return out

known_rows = ~train["target_career_id"].isin(new_careers)
vec_cold = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=15000)
lr_cold = LogisticRegression(C=10, max_iter=300).fit(vec_cold.fit_transform(tagged(train[known_rows])), train.loc[known_rows, "target_career_id"])
vec_all = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=15000)
lr_all = LogisticRegression(C=10, max_iter=300).fit(vec_all.fit_transform(tagged(train)), train["target_career_id"])

rows = []
damaged = add_noise(test, DAMAGE)
is_new = damaged["target_career_id"].isin(new_careers).to_numpy()
for name, scores in (("Starting model (not fine-tuned)", rank_scores(base, damaged)),
                     ("Logistic Regression", lr_scores(lr_cold, vec_cold, damaged)),
                     ("Career matcher (fine-tuned)", rank_scores(held_out_model, damaged))):
    for group, mask in (("careers seen in training", ~is_new), ("new careers", is_new)):
        hit, mrr = hit_and_mrr(scores[mask], damaged[mask])
        rows.append({"model": name, "careers": group, "hit@1": hit, "mrr": mrr})
held_out = pd.DataFrame(rows)
held_out.to_csv(RESULTS / "new_careers.csv", index=False)
print(held_out.to_string(index=False))

## Final model for the app

The model used by the backend is trained on all careers. Its accuracy is checked on the damaged test profiles at two levels of damage, next to Logistic Regression trained on the same data.

In [ ]:
final_model = train_matcher()
rows = []
for level in (0.5, 0.7):
    damaged = add_noise(test, level)
    for name, scores in (("Logistic Regression", lr_scores(lr_all, vec_all, damaged)),
                         ("Career matcher (fine-tuned)", rank_scores(final_model, damaged))):
        hit, mrr = hit_and_mrr(scores, damaged)
        rows.append({"model": name, "damaged": level, "hit@1": hit, "mrr": mrr})
all_careers_results = pd.DataFrame(rows)
all_careers_results.to_csv(RESULTS / "all_careers.csv", index=False)
print(all_careers_results.to_string(index=False))

final_model.save(str(OUT_DIR))
json.dump({"base_model": BASE_MODEL, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
           "max_tokens": MAX_TOKENS, "trained_pairs": 2 * len(train)}, open(OUT_DIR / "training_settings.json", "w"), indent=2)
print("Saved to", OUT_DIR)

## Download

Unzip `career_matcher_model.zip` into `ml/models/` in the project, so that the folder `ml/models/career-matcher` exists. The backend picks it up the next time it starts.

In [ ]:
import shutil
archive = shutil.make_archive(str(PROJECT / "career_matcher_model"), "zip", root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name)
print("Created", archive)
if IN_COLAB:
    files.download(archive)